In [99]:
import pandas as pd
import numpy as np

In [173]:
AIR_CSV = "../opensky_2022/states_2022-01-03-00.csv/states_2022-01-03-00.csv"
df = pd.read_csv(AIR_CSV)
df = df.drop(columns=["lastcontact", "lastposupdate", "callsign", "squawk"])

In [174]:
total_flights = df['icao24'].unique()  
flights_with_nan = df[df.isnull().any(axis=1)]['icao24'].unique()

print(f"Total flights: {len(total_flights)}")
print(f"Total flights with at least one NaN: {len(flights_with_nan)}")
print(f"There is a {len(flights_with_nan) / len(total_flights)*100:.2f}% of missing flights")

Total flights: 10508
Total flights with at least one NaN: 5255
There is a 50.01% of missing flights


In [175]:
nan_analysis = []

grouped = df.sort_values(['icao24', 'time']).groupby('icao24')

n_cols = len(df.columns)
for icao, group in grouped:
    if icao in flights_with_nan:
        n_rows = len(group)
        null_counts = group.isnull().sum()
        cols_with_nan = null_counts[null_counts > 0].index.tolist()
        
        any_nan_mask = group.isnull().any(axis=1).values
        rows_affected_count = any_nan_mask.sum()
        
        nan_analysis.append({
            'icao24': icao,
            'total_rows': n_rows,
            'rows_with_nans': rows_affected_count,
            "row_nan_pct": round((rows_affected_count / n_rows) * 100 , 2),
            'columns_affected': cols_with_nan,
        })

# Convert to DataFrame for easier inspection
nan_summary_df = pd.DataFrame(nan_analysis)

In [176]:
nan_summary_df[nan_summary_df["row_nan_pct"] < 10]["rows_with_nans"].value_counts()

rows_with_nans
1     1454
2      404
3      207
4      128
6       90
5       86
7       62
8       51
11      38
10      34
9       25
12      23
17      22
14      21
19      21
13      19
15      18
18      15
27      12
30      12
20      10
16       8
21       8
23       8
22       8
25       7
26       6
31       6
33       4
28       3
29       3
24       3
32       2
34       1
35       1
Name: count, dtype: int64

In [177]:
nan_summary_df[nan_summary_df["row_nan_pct"] < 10]["columns_affected"].value_counts()


columns_affected
[lat, lon, baroaltitude, geoaltitude]                                 1305
[geoaltitude]                                                          474
[lat, lon, velocity, heading, vertrate, baroaltitude, geoaltitude]     275
[velocity, heading, vertrate, geoaltitude]                             224
[lat, lon]                                                             182
[baroaltitude, geoaltitude]                                            142
[lat, lon, velocity, heading, vertrate, geoaltitude]                    89
[lat, lon, geoaltitude]                                                 79
[velocity, heading, vertrate, baroaltitude, geoaltitude]                49
[baroaltitude]                                                           1
Name: count, dtype: int64

In [178]:
print("\n--- Summary of flights with NaNs ---")
display(nan_summary_df[nan_summary_df["row_nan_pct"] < 10])

nan_col_freq = df.isnull().sum().sort_values(ascending=False)
print("\n--- NaN counts per column ---")
display(nan_col_freq[nan_col_freq > 0])


--- Summary of flights with NaNs ---


,icao24,total_rows,rows_with_nans,row_nan_pct,columns_affected
6,01013c,282,1,0.35,"[lat, lon]"
7,0101bd,89,1,1.12,"[lat, lon, baroaltitude, geoaltitude]"
8,0101df,248,1,0.40,"[lat, lon, geoaltitude]"
9,010205,45,1,2.22,"[lat, lon, baroaltitude, geoaltitude]"
10,010207,42,3,7.14,"[lat, lon, baroaltitude, geoaltitude]"
...,...,...,...,...,...
5213,e49645,137,2,1.46,[geoaltitude]
5215,e49686,215,1,0.47,"[lat, lon, baroaltitude, geoaltitude]"
5216,e497e6,219,6,2.74,"[lat, lon, velocity, heading, vertrate, baroal..."
5217,e497fe,143,11,7.69,"[lat, lon, baroaltitude, geoaltitude]"



--- NaN counts per column ---


geoaltitude     247252
baroaltitude    177814
heading         173058
velocity        173058
vertrate        172831
lon             144435
lat             144435
dtype: int64

In [179]:
len(nan_summary_df[nan_summary_df["row_nan_pct"] < 10]["icao24"].unique())

2820

In [180]:
df[df["icao24"] == "e80326"].iloc[45:47]

,time,icao24,lat,lon,velocity,heading,vertrate,onground,alert,spi,baroaltitude,geoaltitude
348028,1641168590,e80326,NaN,NaN,NaN,NaN,NaN,False,False,False,NaN,NaN
412758,1641168700,e80326,NaN,NaN,NaN,NaN,NaN,False,False,False,NaN,NaN


In [ ]:
df[df["icao24"] == "e80326"].iloc[0:84]

,time,icao24,lat,lon,velocity,heading,vertrate,onground,alert,spi,baroaltitude,geoaltitude
73400,1641168140,e80326,NaN,NaN,NaN,NaN,NaN,False,False,False,NaN,NaN
82530,1641168150,e80326,NaN,NaN,NaN,NaN,NaN,False,False,False,NaN,NaN
84268,1641168160,e80326,NaN,NaN,NaN,NaN,NaN,False,False,False,NaN,NaN
90681,1641168170,e80326,NaN,NaN,NaN,NaN,NaN,False,False,False,NaN,NaN
99481,1641168180,e80326,NaN,NaN,NaN,NaN,NaN,False,False,False,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
613913,1641169030,e80326,NaN,NaN,NaN,NaN,NaN,False,False,False,NaN,NaN
621396,1641169040,e80326,NaN,NaN,NaN,NaN,NaN,False,False,False,NaN,NaN
630730,1641169050,e80326,NaN,NaN,NaN,NaN,NaN,False,False,False,NaN,NaN
1417365,1641170390,e80326,NaN,NaN,81.029421,169.760225,2.92608,False,False,False,NaN,NaN


In [182]:
df[df["icao24"] == "0101df"].iloc[201:203]

,time,icao24,lat,lon,velocity,heading,vertrate,onground,alert,spi,baroaltitude,geoaltitude
1201307,1641170020,0101df,43.433945,16.6135,274.450932,128.760445,0.00000,False,False,False,12489.18,12702.54
1846518,1641171140,0101df,NaN,NaN,260.506843,136.120166,-0.32512,False,False,False,12496.80,NaN


In [183]:
df[df["icao24"] == "0100e5"].iloc[212:214]

,time,icao24,lat,lon,velocity,heading,vertrate,onground,alert,spi,baroaltitude,geoaltitude
1267060,1641170130,0100e5,34.653905,27.777901,239.217013,139.011126,-0.32512,False,False,False,11285.22,11437.62
1994348,1641171400,0100e5,32.154050,30.461276,235.281513,137.835863,0.00000,False,False,False,11285.22,11407.14


In [184]:
df[df["icao24"] == "01013c"]

,time,icao24,lat,lon,velocity,heading,vertrate,onground,alert,spi,baroaltitude,geoaltitude
239539,1641168410,01013c,NaN,NaN,96.096432,299.859016,13.00480,False,False,False,228.60,160.02
243526,1641168420,01013c,40.649780,-73.820740,97.968833,289.957402,11.70432,False,False,False,358.14,289.56
250959,1641168430,01013c,40.652115,-73.839355,103.100790,261.101687,10.40384,False,False,False,457.20,396.24
256460,1641168440,01013c,40.650284,-73.845520,108.033240,233.130102,10.72896,False,False,False,563.88,495.30
262849,1641168450,01013c,40.640808,-73.859253,110.250762,223.676571,10.40384,False,False,False,670.56,601.98
...,...,...,...,...,...,...,...,...,...,...,...,...
2079242,1641171550,01013c,44.923841,-65.564263,338.228836,64.214435,0.00000,False,False,False,10058.40,9944.10
2085366,1641171560,01013c,44.923841,-65.564263,338.228836,64.214435,0.00000,False,False,False,10058.40,9944.10
2090000,1641171570,01013c,44.923841,-65.564263,338.228836,64.214435,0.00000,False,False,False,10058.40,9944.10
2095371,1641171580,01013c,44.923841,-65.564263,338.228836,64.214435,0.00000,False,False,False,10058.40,9944.10


In [185]:
import pandas as pd

# 1. Identify Gaps & Create Groups (using temporary series to keep df clean)
# We assume a gap is anything NOT equal to 10 seconds
time_diffs = df.groupby('icao24')['time'].diff()
is_gap = (time_diffs.fillna(10) != 10)
sequence_grp = is_gap.groupby(df['icao24']).cumsum()

# 2. Update the icao24 column in place
# We use the temporary sequence_grp so we don't store it in the df
df['icao24'] = df['icao24'].astype(str) + '.' + sequence_grp.astype(str)

# 3. Build the Analysis Report
# We derive the 'base_icao' from the new icao24 for the report
report = df.assign(
    base_icao=df['icao24'],
    diffs=time_diffs # temporary column inside assign only
).groupby('base_icao').agg(
    data_start=('time', 'min'),
    data_end=('time', 'max'),
    total_gaps=('icao24', lambda x: x.str.split('.').str[1].astype(int).max()),
    max_gap_seconds=('diffs', 'max'),
    gap_values=('diffs', lambda x: list(x[x.notnull() & (x != 10)].unique())),
    segment_lengths=('icao24', lambda x: list(df.loc[x.index].groupby('icao24').size()))
).reset_index()

# 4. Final Cleanup
# If you don't want 'time_diff' in your final df, don't assign it in step 1
# But if it's already there, we drop it now.
df.drop(columns=[col for col in ['is_gap', 'sequence_grp', 'time_diff'] if col in df.columns], inplace=True)

print("Cleaned DataFrame:")
print(df.head())
print("\nGap Analysis Report:")

Cleaned DataFrame:
         time    icao24        lat        lon    velocity     heading  \
0  1641168000  4ca8e8.0  52.980357   0.948434  193.792496  284.921620   
1  1641168000  471f41.0  42.852219   5.691572  239.283384   55.865788   
2  1641168000  aa56d4.0  38.396844 -77.004905  288.196102   39.059131   
3  1641168000  3c4582.0  52.690109  -1.461182  158.457103  149.355259   
4  1641168010  a4ee2e.0  40.516694 -76.364906  164.960141  281.695403   

   vertrate  onground  alert    spi  baroaltitude  geoaltitude  
0   0.00000     False  False  False      11582.40     11330.94  
1   0.32512     False  False  False      10972.80     11300.46  
2   0.00000     False  False  False      11277.60     11422.38  
3  26.00960     False  False  False       3695.70      3604.26  
4   5.20192     False  False  False       8168.64      8168.64  

Gap Analysis Report:


In [186]:
report[report["total_gaps"] > 0]

,base_icao,data_start,data_end,total_gaps,max_gap_seconds,gap_values,segment_lengths
7,0100e5.1,1641171400,1641171590,1,1270.0,[1270.0],[20]
10,01013c.1,1641171280,1641171590,1,380.0,[380.0],[32]
16,0101df.1,1641171140,1641171590,1,1120.0,[1120.0],[46]
23,01022f.1,1641171420,1641171590,1,70.0,[70.0],[18]
32,02a1cd.1,1641170070,1641170400,1,770.0,[770.0],[34]
...,...,...,...,...,...,...,...
12194,e8032d.1,1641171150,1641171590,1,130.0,[130.0],[45]
12202,e8043e.1,1641170800,1641171210,1,180.0,[180.0],[42]
12207,e80451.1,1641171200,1641171500,1,530.0,[530.0],[31]
12214,e8060a.1,1641169320,1641170990,1,110.0,[110.0],[168]


In [188]:
df

,time,icao24,lat,lon,velocity,heading,vertrate,onground,alert,spi,baroaltitude,geoaltitude
0,1641168000,4ca8e8.0,52.980357,0.948434,193.792496,284.921620,0.00000,False,False,False,11582.40,11330.94
1,1641168000,471f41.0,42.852219,5.691572,239.283384,55.865788,0.32512,False,False,False,10972.80,11300.46
2,1641168000,aa56d4.0,38.396844,-77.004905,288.196102,39.059131,0.00000,False,False,False,11277.60,11422.38
3,1641168000,3c4582.0,52.690109,-1.461182,158.457103,149.355259,26.00960,False,False,False,3695.70,3604.26
4,1641168010,a4ee2e.0,40.516694,-76.364906,164.960141,281.695403,5.20192,False,False,False,8168.64,8168.64
...,...,...,...,...,...,...,...,...,...,...,...,...
2103854,1641171590,a43ef6.0,33.025269,-97.306458,98.800038,8.383248,0.00000,False,False,False,1836.42,1897.38
2103855,1641171590,abca53.0,36.989710,-82.760726,227.223573,321.988490,-0.32512,False,False,False,10363.20,10469.88
2103856,1641171590,a1c486.1,38.136478,-121.606830,53.027674,194.036243,-0.65024,False,False,False,1280.16,1318.26
2103857,1641171590,abbf2e.0,40.699493,-74.173813,70.170237,26.565051,-4.22656,True,False,False,NaN,NaN


In [ ]:
import pandas as pd
import numpy as np

def interpolate_nan_values(df: pd.DataFrame, cols_to_interpolate: list[str], group_col: str = 'icao24') -> pd.DataFrame:
    grouped = df.groupby(group_col)[cols_to_interpolate]
    
    ffill_vals = grouped.ffill()
    diffs = grouped.diff()
    ffill_diffs = diffs.groupby(df[group_col]).ffill()

    for col in cols_to_interpolate:
        is_nan = df[col].isna()
        
        if not is_nan.any():
            continue
            
        block_id = (~is_nan).cumsum()
        
        nan_count = df.groupby([group_col, block_id]).cumcount()
        
        projected = ffill_vals[col] + (nan_count * ffill_diffs[col])
        
        df.loc[is_nan, col] = projected[is_nan]

    return df.dropna().reset_index(drop=True)

In [190]:
def filter_flights_by_nan(df: pd.DataFrame, threshold_pct: float = 10.0) -> pd.DataFrame:
    """
    Drops flight segments where the number of rows containing at least 
    one NaN exceeds the threshold percentage.
    """
    if df.empty:
        return df

    # 1. Identify rows with at least one NaN (matching your analysis logic)
    # any(axis=1) creates the 'any_nan_mask' for the whole dataframe
    any_nan_mask = df.isnull().any(axis=1)
    
    # 2. Group by the split icao24 to calculate stats per segment
    # We sum the mask (True=1) to get rows_affected_count
    nan_stats = any_nan_mask.groupby(df['icao24']).agg(['count', 'sum'])
    nan_stats.columns = ['total_rows', 'rows_with_nans']
    
    # 3. Calculate percentage
    nan_stats['nan_pct'] = (nan_stats['rows_with_nans'] / nan_stats['total_rows']) * 100
    
    # 4. Identify valid segments (less than or equal to threshold)
    valid_segments = nan_stats[nan_stats['nan_pct'] <= threshold_pct].index
    
    print(f"Filtering: Keeping {len(valid_segments)} segments; "
          f"Dropping {len(nan_stats) - len(valid_segments)} segments with >{threshold_pct}% NaNs.")
    
    return df[df['icao24'].isin(valid_segments)].copy()

df_filtered = filter_flights_by_nan(df)

Filtering: Keeping 9385 segments; Dropping 2836 segments with >10.0% NaNs.


In [201]:
df_app = apply_custom_interpolation(df_filtered)

C:\Users\camar\AppData\Local\Temp\ipykernel_2428\2230505653.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby('icao24', group_keys=False).apply(


In [202]:
df_app

,time,icao24,lat,lon,velocity,heading,vertrate,onground,alert,spi,baroaltitude,geoaltitude
717464,1641169200,0003b2.0,24.898651,46.469007,172.138989,127.349349,-4.8768,False,False,False,2933.70,3002.28
727080,1641169210,0003b2.0,24.894833,46.474500,170.600831,127.526455,-4.8768,False,False,False,2910.84,2987.04
727897,1641169220,0003b2.0,24.894833,46.474500,170.600831,127.526455,-4.8768,False,False,False,2910.84,2987.04
735100,1641169230,0003b2.0,24.894833,46.474500,170.600831,127.526455,-4.8768,False,False,False,2910.84,2987.04
743691,1641169240,0003b2.0,24.894833,46.474500,170.600831,127.526455,-4.8768,False,False,False,2910.84,2987.04
...,...,...,...,...,...,...,...,...,...,...,...,...
1740558,1641170950,e8060a.1,-4.948829,-78.460520,225.060872,4.720137,0.0000,False,False,False,11574.78,12329.16
1741804,1641170960,e8060a.1,-4.948829,-78.460520,225.060872,4.720137,0.0000,False,False,False,11574.78,12329.16
1748463,1641170970,e8060a.1,-4.948829,-78.460520,225.060872,4.720137,0.0000,False,False,False,11574.78,12329.16
1757447,1641170980,e8060a.1,-4.948829,-78.460520,225.060872,4.720137,0.0000,False,False,False,11574.78,12329.16


In [203]:
df_app.isna().sum()

time            0
icao24          0
lat             0
lon             0
velocity        0
heading         0
vertrate        0
onground        0
alert           0
spi             0
baroaltitude    0
geoaltitude     0
dtype: int64

In [195]:
import pandas as pd

def get_sequences_with_gaps(df: pd.DataFrame, time_col: str = 'time', group_col: str = 'icao24', step: int = 10) -> pd.DataFrame:
    """
    Identifies which sequences have missing time steps.
    
    Args:
        df: The input DataFrame.
        time_col: The name of the time column.
        group_col: The column defining the sequences (e.g., flight ID).
        step: The expected difference between consecutive rows.
        
    Returns:
        A summary DataFrame containing only the groups that have missing steps, 
        along with the total number of gaps and the maximum gap size observed.
    """
    if df.empty:
        return pd.DataFrame()

    # 1. Sort the data to ensure time flows sequentially within each group
    # (Crucial before calculating differences)
    df_sorted = df.sort_values(by=[group_col, time_col])
    
    # 2. Calculate the time difference between consecutive rows per group
    time_diffs = df_sorted.groupby(group_col)[time_col].diff()
    
    # 3. Create a boolean mask for rows where the gap is larger than expected
    # (The first row of each group will be NaN, which evaluates to False here)
    is_gap = time_diffs > step
    
    # 4. Extract only the rows where a gap occurred
    gap_rows = df_sorted[is_gap].copy()
    gap_rows['gap_size'] = time_diffs[is_gap]
    
    # 5. Aggregate the results to show a clean summary per sequence
    summary = gap_rows.groupby(group_col).agg(
        missing_steps_count=(time_col, 'count'),
        max_gap_size=('gap_size', 'max')
    ).reset_index()
    
    return summary

In [204]:
flights_with_gaps = get_sequences_with_gaps(df_app, time_col='time', group_col='icao24', step=10)

print(f"Found {len(flights_with_gaps)} flights with missing steps.")
display(flights_with_gaps)


Found 1 flights with missing steps.


,icao24,missing_steps_count,max_gap_size
0,ac9af0.0,1,20.0


In [208]:
AIR_CSV = "../opensky_2022/states_2022-01-03-00.csv/states_2022-01-03-00.csv"
df2 = pd.read_csv(AIR_CSV)
df2 = df2.drop(columns=["lastcontact", "lastposupdate", "callsign", "squawk"])

In [210]:
df2[df2["icao24"] == "ac9af0"]

,time,icao24,lat,lon,velocity,heading,vertrate,onground,alert,spi,baroaltitude,geoaltitude
158500,1641168280,ac9af0,40.075425,-112.229462,64.435009,19.592282,-1.62560,False,False,False,1722.12,1775.46
165085,1641168290,ac9af0,40.082199,-112.226379,64.435009,19.592282,-1.62560,False,False,False,1699.26,NaN
170273,1641168300,ac9af0,40.088087,-112.223656,66.538514,18.014694,-2.60096,False,False,False,1676.40,1737.36
176832,1641168310,ac9af0,40.091067,-112.222470,66.919258,16.066401,-2.27584,False,False,False,1661.16,1729.74
183954,1641168320,ac9af0,40.091067,-112.222470,66.919258,16.066401,-2.27584,False,False,False,1661.16,1729.74
...,...,...,...,...,...,...,...,...,...,...,...,...
1080480,1641169810,ac9af0,40.621390,-111.995877,5.751659,169.695154,0.65024,False,False,False,1310.64,1394.46
1086239,1641169820,ac9af0,40.621390,-111.995877,5.751659,169.695154,0.65024,False,False,False,1310.64,1394.46
1092273,1641169830,ac9af0,40.621390,-111.995877,5.751659,169.695154,0.65024,False,False,False,1310.64,1394.46
1096794,1641169840,ac9af0,40.621390,-111.995877,5.751659,169.695154,0.65024,False,False,False,1310.64,1394.46


In [205]:
df_app[df_app["icao24"] == "ac9af0.0"]

,time,icao24,lat,lon,velocity,heading,vertrate,onground,alert,spi,baroaltitude,geoaltitude
158500,1641168280,ac9af0.0,40.075425,-112.229462,64.435009,19.592282,-1.62560,False,False,False,1722.12,1775.46
170273,1641168300,ac9af0.0,40.088087,-112.223656,66.538514,18.014694,-2.60096,False,False,False,1676.40,1737.36
176832,1641168310,ac9af0.0,40.091067,-112.222470,66.919258,16.066401,-2.27584,False,False,False,1661.16,1729.74
183954,1641168320,ac9af0.0,40.091067,-112.222470,66.919258,16.066401,-2.27584,False,False,False,1661.16,1729.74
188317,1641168330,ac9af0.0,40.091067,-112.222470,66.919258,16.066401,-2.27584,False,False,False,1661.16,1729.74
194918,1641168340,ac9af0.0,40.091067,-112.222470,66.919258,16.066401,-2.27584,False,False,False,1661.16,1729.74
202457,1641168350,ac9af0.0,40.091067,-112.222470,66.919258,16.066401,-2.27584,False,False,False,1661.16,1729.74
206740,1641168360,ac9af0.0,40.091067,-112.222470,66.919258,16.066401,-2.27584,False,False,False,1661.16,1729.74
214273,1641168370,ac9af0.0,40.091067,-112.222470,66.919258,16.066401,-2.27584,False,False,False,1661.16,1729.74
220647,1641168380,ac9af0.0,40.091067,-112.222470,66.919258,16.066401,-2.27584,False,False,False,1661.16,1729.74
